# Solutions — Hooks fundamentals

Only look here after you've actually tried the exercises in `hooks_fundamentals.ipynb`.

### LESSON 36 — Exercise

**1. What a Hook is, and how to recognise one.** A Hook is a function that lets a component
use a React feature — memory, context, a connection to something outside React. You recognise
one by its name: Hooks begin with `use`.

**2. Which of the five are Hooks.**

| | | why |
|---|---|---|
| `useState` | Hook | built-in, and named for it |
| `handleChange` | not a Hook | an ordinary event handler |
| `useFetchUser` | Hook | a *custom* one — the `use` prefix says it calls Hooks inside |
| `validate` | not a Hook | a pure function (LESSON 33), no React involved |
| `useId` | Hook | built-in, in React's "Other" category |

The honest answer to "how can you tell" is: **by the name, and that is the point.** You cannot
tell by looking at the call. The convention exists precisely so that the answer is readable.

**3. `validate` versus `useState` — the behavioural difference.** Both are functions called
from inside a component, and there the similarity ends.

`validate(values)` is a pure function (LESSON 33): give it the same input and it returns the
same output, it remembers nothing between calls, and you could call it from a notebook cell —
which is exactly what you did.

`useState` cannot do any of that. It only works while React is rendering a component; it
returns a *different* value on different renders because it is reading memory tied to this
component instance; and calling it changes what React will do next. Call it twice and you get
two independent pieces of state, not the same answer twice.

So: one is a calculation, the other is a connection.

**4. The six that do the work.**

| | for |
|---|---|
| `useState` | remembering a value between renders |
| `useEffect` | connecting to something outside React |
| `useRef` | a value that survives renders without causing one, often a DOM node |
| `useContext` | reading a value from further up the tree |
| `useReducer` | state whose update logic is worth keeping in one place |
| `useMemo` | caching an expensive calculation, once measurement justifies it |

Used so far: `useState`. Next: `useEffect`, in topic 14.

**5. A Hook that reads the current URL.** You may have guessed **Other**, and that is a
reasonable guess — but the real answer is more useful: **there is no such built-in Hook**, and
there is not going to be one. React renders UI; it knows nothing about URLs, history or
navigation. A Hook like that comes from a **router library**, which is topic 20.

That distinction is worth keeping as you read the ecosystem: `useState` and `useEffect` are
React. `useSearchParams` and `useNavigate` are React Router. Both are Hooks and both follow
the same rules, but only one set comes in the box.

**Common mistakes.**

- Deciding a function is a Hook because it is called inside a component. `validate` and
  `handleChange` are both called from components and neither is a Hook.
- Reading the map as a to-do list. Most of those Hooks you will never type, and React marks
  several of them *rarely used*.
- Assuming `useFetchUser` must be something React provides. Anything can be a custom Hook —
  the prefix is a promise about the rules it follows, not about who wrote it.

### LESSON 36 — Mini challenge

**1. What the `use` prefix buys.** JavaScript does not care, and that is the point — the
convention exists for everyone *else*. It tells a reader that this function has rules about
where it may be called (LESSON 37). It tells React's lint rules which calls to check, since
they match on the name. And it tells you, at the call site, that this line cannot be moved
into an `if` without consequences.

Rename `useState` to `createState` and nothing breaks at runtime — but every tool and every
reader loses the ability to spot a misplaced call.

**2. What is wrong with `getFormState`.** The name says "ordinary function"; the body calls
`useState`, which makes it a custom Hook whether it is named like one or not.

Beyond the name, two things go wrong. The lint rules will not check it, because they identify
Hooks by prefix — so a call inside a condition sails through. And the next person to read it
will reasonably assume they can call it anywhere: inside a handler, inside a loop, inside an
`if`. Every one of those is a Rules of Hooks violation that the name actively invited.

Name it `useFormState` and both problems disappear.

**3. Is `useState` "just a function"?** What is true: it is a function, you call it, it
returns a value.

What is missing: it only works while React is rendering a component; it reads memory that
belongs to that component instance rather than computing an answer; calling it can cause React
to render again; and **where** you call it matters, which is true of no ordinary function. A
plain function does not care whether it was called first or third. `useState` does.

### LESSON 37 — Exercise

**Part 1.**

In [ ]:
// Conceptual model — not React's actual implementation.
function l37makeComponent() {
  const slots = [];
  let cursor = 0;
  return {
    startRender() { cursor = 0; },
    useStateModel(initial) {
      const index = cursor;
      cursor = cursor + 1;
      if (slots.length <= index) slots[index] = initial;
      return [slots[index], (next) => { slots[index] = next; }];
    },
    slots,
  };
}

const l37x = l37makeComponent();

// 1 - three Hooks, set them all, re-render in the same order
l37x.startRender();
let [l37n, l37setN] = l37x.useStateModel("Ada");
let [l37c, l37setC] = l37x.useStateModel(0);
let [l37f, l37setF] = l37x.useStateModel(false);
l37setN("Grace"); l37setC(7); l37setF(true);

l37x.startRender();
[l37n] = l37x.useStateModel("Ada");
[l37c] = l37x.useStateModel(0);
[l37f] = l37x.useStateModel(false);
console.log("1 same order  ->", l37n, l37c, l37f, "| correct");

// 2 - the SECOND call is skipped this time
l37x.startRender();
[l37n] = l37x.useStateModel("Ada");      // slot 0 - still the name
[l37f] = l37x.useStateModel(false);      // slot 1 - this is the COUNT's slot
console.log("2 second skipped ->", "name:", l37n, "| flag:", l37f, "<- flag read the count's slot");

// Slot 0 was the name, so the name is still right. The flag asked for slot 1, which holds
// the count, so `flag` is now 7. Everything after the skipped call is shifted by one.

// 3 - a FOURTH call appears on a later render
l37x.startRender();
[l37n] = l37x.useStateModel("Ada");
[l37c] = l37x.useStateModel(0);
[l37f] = l37x.useStateModel(false);
const [l37extra] = l37x.useStateModel("new");
console.log("3 extra Hook  ->", l37extra, "| slots now:", JSON.stringify(l37x.slots));

// The model just grows a fourth slot and carries on - it has no idea anything is wrong.
// React does: it compares the number of Hooks with the previous render and throws
// "Rendered more hooks than during the previous render."

The model is deliberately naive: it cannot tell a legitimate fourth Hook from a bug, which is
exactly why React counts.

**Part 2 — the four snippets.**

In [ ]:
// A - BREAKS the rule. "Do not call Hooks after a conditional `return` statement."
//     When `user` is falsy the component returns early and useState is never reached, so
//     that render makes ZERO Hook calls where the previous one made one.
//     React: "Rendered fewer hooks than expected. This may be caused by an accidental
//     early return statement."
//     The fix is ordering, not logic: the Hook goes above the early return.
//
// B - BREAKS the rule, twice over. The call is inside a loop (`.map`) and inside a nested
//     function (the callback). The number of Hook calls now depends on `items.length`, so
//     every render with a different number of items is a different Hook sequence.
//     This is also the wrong shape: state belonging to a row belongs in a Row component,
//     one instance each (LESSON 25).
//
// C - BREAKS the rule. "Do not call Hooks in event handlers." The handler does not run
//     during rendering at all, so there is no component render for the Hook to attach to.
//     React: "Invalid hook call. Hooks can only be called inside of the body of a function
//     component."
//
// D - Does NOT break any rule. Both Hooks are called unconditionally at the top level; only
//     the ARGUMENT is conditional, and React does not care what you pass.
//     It is still a bug, just a different one: the initial value is used only on the first
//     render (LESSON 25), so `label` is fixed at "Off" forever and never follows `on`.
//     `label` is a DERIVED value and should not be state at all (LESSON 29):
//         const label = on ? "On" : "Off";
//
// The point of D: "it uses a ternary near a Hook" is not the test. The test is whether the
// Hook CALL happens every render, in the same position.

console.log("A: after early return  - breaks the rule");
console.log("B: inside a loop       - breaks the rule");
console.log("C: in an event handler - breaks the rule");
console.log("D: legal, but derived state - a LESSON 29 bug, not a Hooks bug");

**Common mistakes.**

- Calling D a violation because a condition appears on the line. The rule is about whether the
  *call* happens, not about what is inside the parentheses.
- Thinking B is fine because the list "usually" has the same length. Usually is not always,
  and the failure is silent until React notices the count changed.
- Fixing A by wrapping the Hook in a condition instead of moving it above the return.
- Reading "Invalid hook call" and going straight to the version-mismatch cause. It is listed,
  but breaking the rules is far more common.

### LESSON 37 — Mini challenge

**1. "My condition is always true in practice."** Three answers, strongest last.

It is not always true — it is true until someone passes a different prop, and that someone is
usually you, six months later, with no memory of this constraint.

The failure is not a crash you would notice. When the order shifts, Hooks quietly read each
other's slots: the symptom is a wrong *value*, somewhere else, which is the most expensive
kind of bug to track down.

And the argument proves too much. If the condition is genuinely always true, it is doing
nothing — so remove it and the rule is satisfied for free. A condition worth keeping is one
that can be false.

**2. Why React cannot match Hooks by name.** It never sees the name. `const [count, setCount]
= useState(0)` destructures the returned array into two local variables inside *your*
function; React returns an array and has no idea what you did with it. From React's side the
only fact available is "this was the *n*th Hook call of this component during this render" —
so position is all it has to identify state by.

**3. Snippet A's message.** *"Rendered fewer hooks than expected. This may be caused by an
accidental early return statement."*

The wording points straight at the bug because the second sentence names the cause outright.
That is unusually kind for an error message, and worth remembering: when you see it, look for
a `return` above a Hook rather than reading the whole component.

**4. State needed only when a prop is present.** Call the Hook unconditionally and let the
*value* be conditional — or, better, move the part that needs the state into its own component
and render that component only when the prop exists. The second is usually the right answer,
because a component that only exists when the data exists is simpler than one that has to
handle both cases.

### LESSON 38 — Exercise

In [ ]:
// 1. One click on +1, in three steps.
//
//    TRIGGER  The click runs the handler, which calls setCount. Updating state queues a
//             render. Nothing has been called and nothing drawn yet.
//    RENDER   React calls Experiment15 again. It returns JSX containing <CountDisplay> and
//             <ShoppingList>, so React calls those two functions as well. All three produce
//             descriptions. The screen has not changed.
//    COMMIT   React compares the new description with what is on the page. CountDisplay's
//             text differs, so that text node is updated. ShoppingList produced identical
//             output, so nothing is applied there.
//    Then the browser repaints and the new number is visible.
//
// 2. Which step is responsible:
//    - a component's function body runs .................. RENDER
//    - useState returns the new value ..................... RENDER (during the call)
//    - the text inside a <b> element changes .............. COMMIT
//    - setCount(5) on a counter already at 5, nothing ..... TRIGGER - it never happens.
//         React compares with Object.is, finds no change, and does not queue a render
//         (LESSON 25). No render, so no commit.
//    - the browser repaints ............................... after COMMIT, and it is the
//         browser's work rather than React's
//
// 3. Not a bug. The component was rendered, the output matched what was already there, and
//    so the commit had nothing to apply - which is the normal case, not a failure.
//
// 4. "React re-rendered the whole page, so it rebuilt the whole DOM."
//    The first half can be true: React may well call every component function again.
//    The second half is wrong. Rendering produces a description; committing applies only the
//    differences - "React only changes the DOM nodes if there's a difference between
//    renders." Calling every component is not rebuilding every element.

console.log("trigger -> render -> commit -> paint");

**Common mistakes.**

- Treating "re-render" as meaning "the DOM was rebuilt". That conflates steps 2 and 3, and it
  is the reason people fear renders that cost nothing.
- Putting the `Object.is` bail-out in the render step. It happens earlier: no render is even
  queued.
- Thinking the browser paint is part of React. React finishes at commit; painting is the
  browser's job afterwards.

### LESSON 38 — Mini challenge

**The diagnosis order,** and what each check rules out:

```text
1. TRIGGER  Did a state update actually happen?
            - did the handler run at all?
            - was a setter called?
            - was the new value different from the old one (Object.is)?
            Rules out: "React never even knew about this."

2. RENDER   Was the component called, and did it produce what you expect?
            - is the value you are reading the one you think it is?
            - is the component reading state it does not own?
            Rules out: "React ran, but your logic produced the old answer."

3. COMMIT   Did the output differ from last time?
            - if the description is identical, nothing is applied, correctly
            Rules out: "everything worked and there was genuinely nothing to change."
```

**1. Which step first, and why.** Trigger — because it is the cheapest to check and it
eliminates the largest class of causes. If no setter ran, or it ran with an equal value,
nothing downstream can possibly have happened, and investigating the render is wasted effort.

**2. The handler runs and the screen still does not change.** Trigger is *not* fully
eliminated: the handler running is not the same as a setter running, and a setter called with
an `Object.is`-equal value queues nothing. What the log does eliminate is "the click never
reached your code" — a wiring problem such as LESSON 22's `onClick={fn()}`.

So the suspects are: a setter that was not called or was called with an equal value, a render
producing the same output, or state being read from somewhere other than where it was set.

**3. Two ways a render produces "nothing visible".**

- The component genuinely returned the same description, so the commit had nothing to apply.
- The component returned something different, but you are looking at a different component
  from the one that changed — the value moved somewhere you are not watching.

Telling them apart is topic 11's job. The **Components tab** (LESSON 34) shows you the state
and props actually held right now, which settles whether the data changed at all; and the
**Profiler** (LESSON 35) shows you which components were part of the commit. If the data
changed and the component committed but the screen did not move, you are looking at the wrong
element.